In [7]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd
import os
import time
from concurrent.futures import ThreadPoolExecutor

# 1. Konfigurasi Path
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up'
CSV_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv'
LOG_FILE = os.path.join(OUTPUT_DIR, 'harvesting_errors.log')

# Pastikan folder ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Inisialisasi Client
# Kita gunakan Client yang sama agar koneksi efisien
client_bmkg = Client("https://geof.bmkg.go.id")
client_es = Client("https://service.iris.edu")

# Load Dataset yang sudah memiliki 'nearest_station'
df = pd.read_csv(CSV_PATH).dropna(subset=['nearest_station'])

def download_gempa(row):
    # Nama file unik
    file_name = f"{row['id']}_{row['nearest_station']}.mseed"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    
    # Fitur Resume: Jika file sudah ada, jangan unduh ulang
    if os.path.exists(file_path):
        return f"Skipped: {row['id']}"

    try:
        t = UTCDateTime(row['time'])
        # Pilih server berdasarkan network
        client = client_es if row['network_code'] == "II" else client_bmkg
        
        # Unduh 3 komponen (Z, N, E)
        st = client.get_waveforms(row['network_code'], row['nearest_station'], "*", "BH?", t-60, t+300)
        
        # Validasi 3 komponen
        if len(st) >= 3:
            st.write(file_path, format="MSEED")
            return f"Success: {row['id']}"
        else:
            return f"Incomplete: {row['id']} (Trace: {len(st)})"
            
    except Exception as e:
        with open(LOG_FILE, "a") as f:
            f.write(f"{row['id']} | {row['nearest_station']} | Error: {str(e)}\n")
        return f"Error: {row['id']}"

# 3. Eksekusi Paralel
print(f"Memulai harvesting {len(df)} gempa ke: {OUTPUT_DIR}")
with ThreadPoolExecutor(max_workers=5) as executor:
    # Menggunakan list() untuk menjalankan map dan memproses antrean
    list(executor.map(download_gempa, [row for _, row in df.iterrows()]))

print("Harvesting massal selesai. Silakan cek folder output Anda.")

Memulai harvesting 0 gempa ke: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up
Harvesting massal selesai. Silakan cek folder output Anda.


In [6]:
import pandas as pd

# Muat file
df = pd.read_csv('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv')

print(f"Total baris dalam CSV: {len(df)}")
print("Jumlah data dengan 'nearest_station' yang terisi:")
print(df['nearest_station'].notna().sum())
print("\nContoh 5 baris pertama kolom 'nearest_station':")
print(df['nearest_station'].head())

Total baris dalam CSV: 6543
Jumlah data dengan 'nearest_station' yang terisi:
0

Contoh 5 baris pertama kolom 'nearest_station':
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: nearest_station, dtype: float64


In [8]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# 1. Pilih 1 gempa dari data Anda untuk dites
sample_row = df.iloc[0] 
t = UTCDateTime(sample_row['time'])
net = sample_row['network_code']
sta = sample_row['nearest_station']

print(f"Mencoba mengunduh ID: {sample_row['id']}")
print(f"Target: Net={net}, Sta={sta}, Waktu={t}")

# 2. Tes koneksi manual
try:
    client = Client("https://service.iris.edu") if net == "II" else Client("https://geof.bmkg.go.id")
    st = client.get_waveforms(net, sta, "*", "BH?", t-60, t+300)
    print("✅ BERHASIL! Data ditemukan:")
    print(st)
except Exception as e:
    print(f"❌ GAGAL. Pesan error: {e}")

IndexError: single positional indexer is out-of-bounds

In [9]:
# 1. Baca ulang file asli tanpa filter
df_raw = pd.read_csv('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv')

print(f"Total baris dalam file: {len(df_raw)}")
# Cek berapa banyak yang benar-benar terisi
filled_count = df_raw['nearest_station'].notna().sum()
print(f"Jumlah baris dengan 'nearest_station' terisi: {filled_count}")

# 2. Jika filled_count adalah 0, kita harus menjalankan ulang pencarian stasiun
# (Jangan jalankan harvesting jika filled_count masih 0!)

Total baris dalam file: 6543
Jumlah baris dengan 'nearest_station' terisi: 0


In [11]:
import pandas as pd
from obspy import read_inventory
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'balanced_dataset.csv'))
master_inv = read_inventory(os.path.join(OUTPUT_DIR, 'master_station_inventory.xml'))

# Cek 1 data gempa pertama
lat_gempa = df.loc[0, 'latitude']
lon_gempa = df.loc[0, 'longitude']

# Cek 1 data stasiun pertama
stasiun_contoh = master_inv[0][0]
lat_sta = stasiun_contoh.latitude
lon_sta = stasiun_contoh.longitude

print(f"Koordinat Gempa Pertama: {lat_gempa}, {lon_gempa}")
print(f"Koordinat Stasiun Pertama: {lat_sta}, {lon_sta}")

Koordinat Gempa Pertama: -3.38, 129.89
Koordinat Stasiun Pertama: -7.6865, 108.67377


In [12]:
from obspy.clients.fdsn import Client
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
TARGET_PATH = os.path.join(OUTPUT_DIR, 'master_station_inventory_V2.xml')

# Gunakan cakupan seluruh Indonesia tanpa batas koordinat yang sempit
print("Mengambil data stasiun seluruh jaringan Indonesia...")
bmkg = Client("https://geof.bmkg.go.id")
iris = Client("https://service.iris.edu")

# Menarik data dari seluruh jaringan yang tersedia di wilayah Indonesia
inv_bmkg = bmkg.get_stations(network="IA,VG") 
inv_iris = iris.get_stations(network="II")

master_inv = inv_bmkg + inv_iris
master_inv.write(TARGET_PATH, format="STATIONXML")

print(f"✅ Database stasiun diperbarui. Total stasiun: {sum(len(n) for n in master_inv)}")

Mengambil data stasiun seluruh jaringan Indonesia...
✅ Database stasiun diperbarui. Total stasiun: 1420


In [13]:
import pandas as pd
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'dataset_with_stations.csv'))
print(f"Stasiun terisi: {df['nearest_station'].notna().sum()} dari {len(df)} gempa.")

Stasiun terisi: 0 dari 6543 gempa.


In [14]:
from obspy import read_inventory
import pandas as pd
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
master_inv = read_inventory(os.path.join(OUTPUT_DIR, 'master_station_inventory_V2.xml'))
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'balanced_dataset.csv'))

# Ambil satu stasiun dan satu gempa
stasiun_contoh = master_inv[0][0]
gempa_contoh = df.iloc[0]

print(f"Lat Stasiun: {stasiun_contoh.latitude} (Tipe: {type(stasiun_contoh.latitude)})")
print(f"Lat Gempa: {gempa_contoh['latitude']} (Tipe: {type(gempa_contoh['latitude'])})")

Lat Stasiun: -7.6865 (Tipe: <class 'obspy.core.inventory.util.Latitude'>)
Lat Gempa: -3.38 (Tipe: <class 'numpy.float64'>)


In [15]:
from scipy.spatial import KDTree
import pandas as pd
import os
import numpy as np

# 1. Path
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output'
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'balanced_dataset.csv'))

# 2. Ekstrak koordinat stasiun dengan konversi PAKSA ke float
coords_sta = []
info_sta = []
for net in master_inv:
    for sta in net:
        # PENTING: Konversi ke float biasa
        lat = float(sta.latitude)
        lon = float(sta.longitude)
        coords_sta.append([lat, lon])
        info_sta.append((sta.code, net.code))

# 3. Buat KDTree
tree = KDTree(coords_sta)

# 4. Cari tetangga terdekat untuk seluruh gempa sekaligus
coords_gempa = df[['latitude', 'longitude']].values
dist, indices = tree.query(coords_gempa, k=1)

# 5. Masukkan hasil ke DataFrame
df['nearest_station'] = [info_sta[i][0] for i in indices]
df['network_code'] = [info_sta[i][1] for i in indices]

# 6. Simpan
df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_with_stations.csv'), index=False)
print(f"✅ Selesai! Jumlah stasiun terisi: {df['nearest_station'].notna().sum()}")

✅ Selesai! Jumlah stasiun terisi: 6543


In [16]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor

# 1. Konfigurasi Path (Sesuai direktori Anda)
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up'
CSV_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv'
LOG_FILE = os.path.join(OUTPUT_DIR, 'harvesting_errors.log')

# Pastikan direktori ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Inisialisasi Client
client_bmkg = Client("https://geof.bmkg.go.id")
client_es = Client("https://service.iris.edu")

# Load Dataset yang sudah memiliki 'nearest_station'
df = pd.read_csv(CSV_PATH)

def download_gempa(row):
    # Nama file unik berdasarkan ID gempa dan stasiun
    file_name = f"{row['id']}_{row['nearest_station']}.mseed"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    
    # Fitur Resume: Jika file sudah ada, jangan unduh ulang
    if os.path.exists(file_path):
        return f"Skipped: {row['id']}"

    try:
        t = UTCDateTime(row['time'])
        # Pilih server berdasarkan network
        client = client_es if row['network_code'] == "II" else client_bmkg
        
        # Unduh 3 komponen (Z, N, E) dengan durasi 6 menit (t-60 sampai t+300)
        st = client.get_waveforms(row['network_code'], row['nearest_station'], "*", "BH?", t-60, t+300)
        
        # Validasi: Pastikan ada minimal 3 trace (Z, N, E)
        if len(st) >= 3:
            st.write(file_path, format="MSEED")
            return f"Success: {row['id']}"
        else:
            return f"Incomplete: {row['id']} (Trace count: {len(st)})"
            
    except Exception as e:
        # Mencatat error ke file log
        with open(LOG_FILE, "a") as f:
            f.write(f"{row['id']} | {row['nearest_station']} | Error: {str(e)}\n")
        return f"Error: {row['id']}"

# 3. Eksekusi Paralel (Multithreading)
print(f"Memulai harvesting 6.543 gempa ke: {OUTPUT_DIR}")
print("Proses ini akan berjalan di latar belakang. Silakan pantau folder output.")

# Kita menggunakan 5 thread agar tidak membebani server BMKG
with ThreadPoolExecutor(max_workers=5) as executor:
    # Menggunakan list() untuk menjalankan map dan memproses antrean
    results = list(executor.map(download_gempa, [row for _, row in df.iterrows()]))

print("Harvesting massal selesai!")
print(f"Hasil log error (jika ada) dapat dilihat di: {LOG_FILE}")

Memulai harvesting 6.543 gempa ke: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up
Proses ini akan berjalan di latar belakang. Silakan pantau folder output.
Harvesting massal selesai!
Hasil log error (jika ada) dapat dilihat di: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up/harvesting_errors.log


In [17]:
import os

OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up'

if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    mseed_files = [f for f in files if f.endswith('.mseed')]
    print(f"Jumlah file .mseed yang ditemukan di folder: {len(mseed_files)}")
    if len(mseed_files) > 0:
        print(f"Contoh file: {mseed_files[0]}")
    else:
        print("Folder ada, tapi TIDAK ADA file .mseed di dalamnya.")
else:
    print("Error: Path folder tidak ditemukan sama sekali di sistem Anda.")

Jumlah file .mseed yang ditemukan di folder: 0
Folder ada, tapi TIDAK ADA file .mseed di dalamnya.


In [19]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# Gunakan IRIS sebagai sumber data utama
client = Client("https://service.iris.edu")

try:
    # Mengambil data dari stasiun JAGI (BMKG) melalui IRIS
    st = client.get_waveforms("IA", "JAGI", "*", "BH?", UTCDateTime("2026-05-20T00:00:00"), UTCDateTime("2026-05-20T00:05:00"))
    print("✅ Berhasil mengunduh melalui IRIS!")
    print(st)
except Exception as e:
    print(f"❌ Tetap gagal melalui IRIS: {e}")

❌ Tetap gagal melalui IRIS: No data available for request.
HTTP Status code: 204
Detailed response of server:




In [22]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd
import os
import time

# 1. Path
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_bmkg_usgs_3_up'
CSV_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv'

# 2. Inisialisasi Client (Gunakan IRIS/EarthScope saja, tanpa BMKG)
# Ini adalah akses publik yang paling stabil untuk riset global
client = Client("EARTHSCOPE")

df = pd.read_csv(CSV_PATH)

def download_single_event(row):
    # Nama file
    file_name = f"{str(row['id']).strip()}_{str(row['nearest_station']).strip()}.mseed"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    
    if os.path.exists(file_path):
        return
    
    try:
        t = UTCDateTime(row['time'])
        
        # PENTING: .strip() membersihkan spasi/karakter tersembunyi
        # PENTING: Memastikan tipe data benar
        net = str(row['network_code']).strip()
        sta = str(row['nearest_station']).strip()
        
        # Percobaan request
        st = client.get_waveforms(net, sta, "*", "BH?", t-60, t+300)
        st.write(file_path, format="MSEED")
        
        print(f"✅ Berhasil: {row['id']}")
        time.sleep(3) # Jeda agar tidak diblokir
        
    except Exception as e:
        # Menampilkan detail error agar kita tahu kolom mana yang bermasalah
        print(f"❌ Melewati {row['id']} karena: {e}")

# 3. Jalankan satu per satu (bukan paralel) untuk stabilitas total
print("Mulai mengunduh secara berurutan (Slow Mode)...")
for _, row in df.iterrows():
    download_single_event(row)

Mulai mengunduh secara berurutan (Slow Mode)...
❌ Melewati BMKG-20170507103900-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20221217172936-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20230929052358-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20171014061303-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20120217215023-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20121129072444-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20210203115920-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20220124013804-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20230423153714-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20160712122538-001 karena: 'str' object cannot be interpreted as an integer
❌ Melewati BMKG-20220929231947-0

In [23]:
import pandas as pd
import numpy as np

# 1. Bersihkan CSV dari karakter aneh
df = pd.read_csv('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv')

# Paksa kolom menjadi tipe data yang tepat
df['network_code'] = df['network_code'].astype(str).str.strip()
df['nearest_station'] = df['nearest_station'].astype(str).str.strip()
df['id'] = df['id'].astype(str).str.strip()

# Buang baris yang rusak
df = df.dropna(subset=['network_code', 'nearest_station', 'time'])

print("Cek tipe data kolom:")
print(df[['network_code', 'nearest_station']].dtypes)
print(f"Total baris setelah bersih-bersih: {len(df)}")

Cek tipe data kolom:
network_code       object
nearest_station    object
dtype: object
Total baris setelah bersih-bersih: 6543


In [5]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import pandas as pd
import os
import time
from tqdm import tqdm  # Pustaka untuk progress bar

# 1. Konfigurasi Path
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_iris_only'
CSV_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/dataset_with_stations.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Inisialisasi Client IRIS
client = Client("EARTHSCOPE", user_agent="DoctoralResearch_VeryKurniaBakti")

df = pd.read_csv(CSV_PATH)

# 3. Fungsi Harvesting dengan Error Handling
def download_iris_data(row):
    file_name = f"{row['id']}_{row['nearest_station']}.mseed"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    
    # Skip jika sudah ada
    if os.path.exists(file_path):
        return True
    
    try:
        t = UTCDateTime(pd.to_datetime(row['time']))
        st = client.get_waveforms(
            network=str(row['network_code']), 
            station=str(row['nearest_station']), 
            location="*", 
            channel="BH?", 
            starttime=t-60, 
            endtime=t+300
        )
        st.write(file_path, format="MSEED")
        return True
    except Exception:
        return False

# 4. Eksekusi dengan Progress Bar
print(f"📡 Memulai Harvesting via IRIS untuk {len(df)} gempa...")

# Menggunakan tqdm untuk membungkus iterasi
success_count = 0
with tqdm(total=len(df), unit="gempa") as pbar:
    for _, row in df.iterrows():
        status = download_iris_data(row)
        if status:
            success_count += 1
        
        pbar.update(1)
        pbar.set_postfix({"Sukses": success_count})
        
        # Jeda sopan agar tidak terblokir
        time.sleep(1.5) 

print(f"\n🏁 Harvesting selesai! Total file terkumpul: {success_count}")

📡 Memulai Harvesting via IRIS untuk 6543 gempa...


  7%|▋         | 435/6543 [15:39<3:39:53,  2.16s/gempa, Sukses=0]


KeyboardInterrupt: 